# Genetic Wake-Word Search

End-to-end pipeline: dataset → genetic HP search → multi-tier training → ONNX export → benchmark.

## Configuration
Set env vars (Kaggle secrets / Paperspace env) or edit the **Config cell** below.

### Dataset modes (mutually exclusive, checked in order)
| Mode | Trigger | Behaviour |
|------|---------|-----------|
| **BYO CSV** | `CUSTOM_TRAIN_CSV` set | Use your own `path,label` CSVs — no TTS/HF |
| **HF override** | `HF_DATASET` set | Force a specific HF repo for positives |
| **Auto** | neither | Known wake words → HF; unknown → TTS |

### Augmentation & negative overrides (all optional, stackable with any mode)
| Variable | Format | Effect |
|----------|--------|--------|
| `NEGATIVES_DIR` | local path | Replace HF general negatives with a local audio dir |
| `EXTRA_NEGATIVES_HF` | `org/repo,...` | Append HF repos to general negatives |
| `BG_NOISE_DIR` | local path | Use local dir for bg-noise augmentation |
| `EXTRA_BG_NOISE_HF` | `org/repo,...` | Append HF repos to bg-noise |
| `MUSIC_DIR` | local path | Use local dir for music augmentation |
| `EXTRA_MUSIC_HF` | `org/repo,...` | Append HF repos to music |
| `RIR_DIR` | local path | Use local dir for RIR augmentation |
| `EXTRA_RIR_HF` | `org/repo,...` | Append HF repos to RIR |

Local-path overrides skip HF downloads for that category.  
HF extras are fetched **in addition to** built-in repos.

### All variables
| Variable | Default | Description |
|----------|---------|-------------|
| `WAKE_WORD` | `hey jarvis` | Wake word phrase |
| `OUTPUT_DIR` | `./ww_output` | Root output directory |
| `LANG_CODE` | `en` | TTS language |
| `N_POSITIVE` | `200` | Positive samples (ignored in BYO CSV mode) |
| `ADVERSARIAL` | `true` | Include adversarial negatives |
| `DOWNLOAD_AUGMENT` | `false` | Download HF bg-noise/music/RIR |
| `CUSTOM_TRAIN_CSV` | _(empty)_ | Path to train CSV (`path,label`) |
| `CUSTOM_TEST_CSV` | _(empty)_ | Path to test CSV (optional; 80/20 split if absent) |
| `HF_DATASET` | _(empty)_ | HF repo ID for positives |
| `NEGATIVES_DIR` | _(empty)_ | Local audio dir → general negatives |
| `EXTRA_NEGATIVES_HF` | _(empty)_ | Extra HF repos for negatives |
| `BG_NOISE_DIR` | _(empty)_ | Local audio dir → bg-noise |
| `EXTRA_BG_NOISE_HF` | _(empty)_ | Extra HF repos for bg-noise |
| `MUSIC_DIR` | _(empty)_ | Local audio dir → music |
| `EXTRA_MUSIC_HF` | _(empty)_ | Extra HF repos for music |
| `RIR_DIR` | _(empty)_ | Local audio dir → RIR |
| `EXTRA_RIR_HF` | _(empty)_ | Extra HF repos for RIR |
| `POPULATION` | `12` | Genetic search population |
| `GENERATIONS` | `5` | Number of generations |
| `EPOCHS_PER_TRIAL` | `3` | Epochs per genetic trial |
| `SEARCH_FULL` | `false` | Full search space (slower) |
| `TIERS_TO_TRAIN` | `micro,small,filterbank_small` | Tiers for final training |
| `FINAL_EPOCHS` | `30` | Epochs for final training |
| `EXPORT_ONNX` | `true` | Export ONNX models |
| `DEVICE` | `auto` | auto / cpu / cuda / mps |
| `SEED` | `42` | Random seed |

In [ ]:
import os

WAKE_WORD          = os.environ.get("WAKE_WORD",          "hey jarvis")
OUTPUT_DIR         = os.environ.get("OUTPUT_DIR",         "./ww_output")
LANG               = os.environ.get("LANG_CODE",          "en")
N_POSITIVE         = int(os.environ.get("N_POSITIVE",     "200"))
ADVERSARIAL        = os.environ.get("ADVERSARIAL",        "true").lower()  == "true"
DOWNLOAD_AUGMENT   = os.environ.get("DOWNLOAD_AUGMENT",   "false").lower() == "true"
# Dataset overrides
CUSTOM_TRAIN_CSV   = os.environ.get("CUSTOM_TRAIN_CSV",   "")
CUSTOM_TEST_CSV    = os.environ.get("CUSTOM_TEST_CSV",    "")
HF_DATASET         = os.environ.get("HF_DATASET",         "")
# Negative / augmentation overrides
NEGATIVES_DIR      = os.environ.get("NEGATIVES_DIR",      "")
EXTRA_NEGATIVES_HF = os.environ.get("EXTRA_NEGATIVES_HF", "")
BG_NOISE_DIR       = os.environ.get("BG_NOISE_DIR",       "")
EXTRA_BG_NOISE_HF  = os.environ.get("EXTRA_BG_NOISE_HF",  "")
MUSIC_DIR          = os.environ.get("MUSIC_DIR",          "")
EXTRA_MUSIC_HF     = os.environ.get("EXTRA_MUSIC_HF",     "")
RIR_DIR            = os.environ.get("RIR_DIR",            "")
EXTRA_RIR_HF       = os.environ.get("EXTRA_RIR_HF",       "")
# Genetic search
POPULATION         = int(os.environ.get("POPULATION",     "12"))
GENERATIONS        = int(os.environ.get("GENERATIONS",    "5"))
EPOCHS_PER_TRIAL   = int(os.environ.get("EPOCHS_PER_TRIAL", "3"))
SEARCH_FULL        = os.environ.get("SEARCH_FULL",        "false").lower() == "true"
# Final models
TIERS_TO_TRAIN     = os.environ.get("TIERS_TO_TRAIN",     "micro,small,filterbank_small").split(",")
FINAL_EPOCHS       = int(os.environ.get("FINAL_EPOCHS",   "30"))
EXPORT_ONNX        = os.environ.get("EXPORT_ONNX",        "true").lower()  == "true"
DEVICE             = os.environ.get("DEVICE",             "auto")
SEED               = int(os.environ.get("SEED",           "42"))

In [ ]:
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("torch", "torchaudio", "soundfile", "numpy", "scikit-learn",
     "matplotlib", "librosa", "onnx", "onnxruntime", "click", "tqdm")
_pip("ovos-plugin-manager", "ovos-tts-plugin-edge-tts",
     "ovos-vad-plugin-silero", "datasets")
try:
    import ww_trainer
except ImportError:
    _pip("ww_trainer")

import os, sys
_platform = (
    "kaggle"     if os.path.exists("/kaggle")     else
    "paperspace" if os.path.exists("/notebooks")  else
    "colab"      if "google.colab" in sys.modules else
    "local"
)
print(f"Platform: {_platform} | Device: {DEVICE} | Wake word: {WAKE_WORD!r}")

In [ ]:
import sys
from pathlib import Path

# Make nb_dataset importable when running from the notebooks/ directory
sys.path.insert(0, str(Path("__file__").parent if "__file__" in dir() else Path(".")))
from nb_dataset import load_byo, load_generated, apply_local_overrides

if CUSTOM_TRAIN_CSV:
    datagen_result = load_byo(
        train_csv=CUSTOM_TRAIN_CSV, test_csv=CUSTOM_TEST_CSV,
        output_dir=OUTPUT_DIR, seed=SEED,
        negatives_dir=NEGATIVES_DIR,
        bg_noise_dir=BG_NOISE_DIR, music_dir=MUSIC_DIR, rir_dir=RIR_DIR,
    )
    print("Mode: BYO CSV")
else:
    datagen_result = load_generated(
        wake_word=WAKE_WORD, output_dir=OUTPUT_DIR,
        n_positive=N_POSITIVE, lang=LANG,
        adversarial=ADVERSARIAL, download_augmentation=DOWNLOAD_AUGMENT,
        seed=SEED, hf_dataset=HF_DATASET,
        extra_negatives_hf=EXTRA_NEGATIVES_HF,
        extra_bg_noise_hf=EXTRA_BG_NOISE_HF,
        extra_music_hf=EXTRA_MUSIC_HF,
        extra_rir_hf=EXTRA_RIR_HF,
    )
    apply_local_overrides(
        datagen_result,
        negatives_dir=NEGATIVES_DIR,
        bg_noise_dir=BG_NOISE_DIR, music_dir=MUSIC_DIR, rir_dir=RIR_DIR,
        output_dir=OUTPUT_DIR,
        extra_bg_noise_hf=EXTRA_BG_NOISE_HF,
        extra_music_hf=EXTRA_MUSIC_HF,
        extra_rir_hf=EXTRA_RIR_HF,
        download_augmentation=DOWNLOAD_AUGMENT,
    )
    print(f"Mode: {'HF override' if HF_DATASET else 'Auto'}")

n_train = sum(1 for _ in open(datagen_result.train_csv))
n_test  = sum(1 for _ in open(datagen_result.test_csv))
print(f"Train: {n_train} samples | Test: {n_test} samples")

In [ ]:
from ww_trainer.sweep import run_genetic_search

genetic_result = run_genetic_search(
    metadata_csv=str(datagen_result.train_csv),
    population_size=POPULATION,
    generations=GENERATIONS,
    epochs_per_trial=EPOCHS_PER_TRIAL,
    featurizer_type="mfcc",
    device=DEVICE,
    output_dir=str(Path(OUTPUT_DIR) / "genetic"),
    full=SEARCH_FULL,
)
best_hp  = genetic_result["best_config"]
best_f1  = genetic_result["best_score"]
print(f"Best search F1 : {best_f1:.4f}")
print(f"Best config    : {best_hp}")

In [ ]:
import matplotlib.pyplot as plt

history = genetic_result["history"]
gens  = [h["generation"] for h in history]
bests = [h["best"]       for h in history]
avgs  = [h["avg"]        for h in history]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(gens, bests, "o-",  label="Best F1")
ax.plot(gens, avgs,  "s--", label="Avg F1",  alpha=0.7)
ax.set_xlabel("Generation"); ax.set_ylabel("F1 (search)")
ax.set_title(f"Genetic Search Evolution \u2014 {WAKE_WORD!r}")
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "evolution.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.quickstart import QuickstartConfig, _train_from_datagen_result

final_results = []
for tier_name in TIERS_TO_TRAIN:
    tier_cfg = QuickstartConfig(
        wake_word=WAKE_WORD,
        output_dir=Path(OUTPUT_DIR) / f"model_{tier_name}",
        tier=tier_name,
        epochs=FINAL_EPOCHS,
        batch_size=best_hp.get("batch_size", 16),
        lr=best_hp.get("lr", 5e-4),
        export_onnx=EXPORT_ONNX,
        device=DEVICE,
        download_augmentation=False,
        seed=SEED,
    )
    result = _train_from_datagen_result(tier_cfg, datagen_result)
    final_results.append({
        "tier": tier_name,
        "f1":   result.metrics.get("f1", 0.0),
        "onnx": result.best_onnx_path,
        "pt":   result.best_model_path,
    })
    print(f"  {tier_name:20s}  F1={result.metrics.get('f1', 0):.3f}")

In [ ]:
names  = [r["tier"] for r in final_results]
scores = [r["f1"]   for r in final_results]

fig, ax = plt.subplots(figsize=(max(5, len(names) * 1.8), 4))
bars = ax.bar(names, scores, color=plt.cm.tab10.colors[:len(names)])
ax.bar_label(bars, fmt="%.3f", padding=3, fontsize=10)
ax.set_ylim(0, 1.1); ax.set_ylabel("F1")
ax.set_title(f"Tier Comparison \u2014 {WAKE_WORD!r}")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / "tier_comparison.png", dpi=120)
plt.show()

In [ ]:
from ww_trainer.benchmark import run_benchmark, plot_results, save_results
from IPython.display import Image, display

bench_dir = Path(OUTPUT_DIR) / "benchmark"
bench_dir.mkdir(parents=True, exist_ok=True)
bench_report = run_benchmark(device=DEVICE, output_dir=str(bench_dir))
save_results(bench_report, bench_dir)
plot_results(bench_report, bench_dir)

for png in sorted(bench_dir.glob("*.png")):
    display(Image(str(png)))

In [ ]:
from IPython.display import Markdown, display

rows = ["| Tier | F1 | ONNX |", "|------|-----|------|"] + [
    f"| {r['tier']} | {r['f1']:.3f} | {'\u2713' if r['onnx'] and Path(r['onnx']).exists() else '\u2014'} |"
    for r in final_results
]
display(Markdown("\n".join(rows)))
print(f"\nAll outputs saved to: {Path(OUTPUT_DIR).resolve()}")

## Next Steps

- **Resume after crash**: re-run from Cell 4 — dataset is never re-synthesised
- **BYO dataset**: `CUSTOM_TRAIN_CSV=/path/to/metadata.csv` (format: `path,label`)
- **BYO negatives**: `NEGATIVES_DIR=/path/to/neg_audio/`
- **BYO augmentation**: `BG_NOISE_DIR`, `MUSIC_DIR`, and/or `RIR_DIR`
- **Extra HF data**: `EXTRA_NEGATIVES_HF=org/repo1,org/repo2` (or per-category variants)
- **Force HF positives**: `HF_DATASET=org/repo-name`
- **More tiers**: `TIERS_TO_TRAIN=micro,small,medium,large`
- **Deeper search**: `SEARCH_FULL=true`, increase `POPULATION` / `GENERATIONS`
- **Production dataset**: `N_POSITIVE=500`, `DOWNLOAD_AUGMENT=true`

### Resources
- [Quickstart guide](../docs/quickstart.md)
- [Training docs](../docs/training.md)
- [Hardware guide](../docs/hardware_guide.md)
- [All tiers reference](../docs/classifiers.md)
- [Search strategies](../docs/search_strategies.md)